# L5: Optimizing HNSW Search

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
import warnings
warnings.filterwarnings('ignore')

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Loading the collection)</code>:</b> The following code block might take a few minutes to complete.</p>

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient("http://localhost:6333", timeout=600)
client.delete_collection("wands-products")
client.recover_snapshot(
    "wands-products", 
    "https://storage.googleapis.com/deeplearning-course-c1/snapshots/wands-products.snapshot",
)
collection = client.get_collection("wands-products")
collection

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=85988, points_count=42994, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'product_description': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), 'product_name': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=2, max_segment_size=None, memmap_thresh

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## HNSW parameters

In [3]:
collection.config.hnsw_config

HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None)

## Test queries

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import pandas as pd

queries_df = pd.read_csv(
    "shared_data/WANDS/query.csv", 
    sep="\t", 
    index_col="query_id",
)
queries_df["query_embedding"] = model.encode(
    queries_df["query"].tolist()
).tolist()
queries_df.sample(n=5)

,query,query_class,query_embedding
query_id,,,
5,sofa with ottoman,Sectionals,"[0.1204514354467392, 0.0037856001872569323, -0..."
158,led nightstand,Nightstands,"[0.02548239193856716, 0.0796252191066742, -0.0..."
224,twin over full bunk beds cool desins,NaN,"[-0.0641869306564331, 0.006200217641890049, 0...."
448,bed side table,End Tables,"[0.0743933916091919, 0.0132503816857934, -0.04..."
106,bed risers,Bed Accessories,"[-0.031503885984420776, -0.039867814630270004,..."


## ANN search

In [6]:
client.search(
    "wands-products",
    query_vector=models.NamedVector(
        name="product_name",
        vector=model.encode(queries_df.loc[0, "query"])
    ),
    limit=3,
    with_vectors=False,
    with_payload=False,
)

[ScoredPoint(id=7465, version=116, score=0.9198917, payload=None, vector=None, shard_key=None),
 ScoredPoint(id=9234, version=144, score=0.8231318, payload=None, vector=None, shard_key=None),
 ScoredPoint(id=42329, version=661, score=0.8180746, payload=None, vector=None, shard_key=None)]

## kNN search

In [7]:
client.search(
    "wands-products",
    query_vector=models.NamedVector(
        name="product_name",
        vector=model.encode(queries_df.loc[0, "query"])
    ),
    limit=3,
    with_vectors=False,
    with_payload=False,
    search_params=models.SearchParams(
        exact=True,  # Turns on the exact search mode
    ),
)

[ScoredPoint(id=7465, version=116, score=0.9198917, payload=None, vector=None, shard_key=None),
 ScoredPoint(id=9234, version=144, score=0.8231318, payload=None, vector=None, shard_key=None),
 ScoredPoint(id=42329, version=661, score=0.8180746, payload=None, vector=None, shard_key=None)]

### Ground truth

In [10]:
from collections import defaultdict
from ranx import Qrels

knn_qrels_dict = defaultdict(dict)
for id, row in queries_df.iterrows():
    query_id = f"query_{id}"
    
    results = client.search(
        collection_name="wands-products",
        query_vector=models.NamedVector(
            name="product_name", 
            vector=row["query_embedding"]
        ),
        with_vectors=False,
        with_payload=False,
        limit=100,
        search_params=models.SearchParams(
            exact=True,  # enable exact search
        ),
    )
    
    for point in results:
        document_id = f"doc_{point.id}"
        # The conversion to integer is required because ranx expects integers
        knn_qrels_dict[query_id][document_id] = int(point.score * 100)
    
qrels = Qrels(knn_qrels_dict)
type(qrels)

ranx.data_structures.qrels.Qrels

In [11]:
qrels['query_0']

{'doc_7465': 91,
 'doc_9234': 82,
 'doc_42329': 81,
 'doc_24010': 81,
 'doc_18273': 81,
 'doc_18276': 80,
 'doc_25431': 80,
 'doc_18272': 78,
 'doc_36910': 78,
 'doc_18277': 78,
 'doc_19456': 77,
 'doc_24006': 76,
 'doc_40996': 76,
 'doc_18274': 75,
 'doc_18275': 75,
 'doc_24008': 75,
 'doc_18270': 75,
 'doc_24009': 75,
 'doc_26069': 75,
 'doc_42330': 75,
 'doc_31556': 75,
 'doc_4410': 75,
 'doc_7506': 74,
 'doc_6168': 74,
 'doc_4034': 74,
 'doc_26070': 74,
 'doc_28058': 73,
 'doc_18271': 73,
 'doc_26068': 73,
 'doc_15612': 73,
 'doc_18158': 73,
 'doc_6982': 73,
 'doc_12409': 73,
 'doc_28687': 73,
 'doc_2187': 72,
 'doc_251': 72,
 'doc_33689': 72,
 'doc_39461': 72,
 'doc_33690': 71,
 'doc_31557': 71,
 'doc_26071': 71,
 'doc_31555': 70,
 'doc_6167': 70,
 'doc_39429': 70,
 'doc_39428': 69,
 'doc_9207': 69,
 'doc_8994': 69,
 'doc_975': 69,
 'doc_19004': 68,
 'doc_24007': 68,
 'doc_28059': 68,
 'doc_27443': 67,
 'doc_40997': 67,
 'doc_20026': 67,
 'doc_16301': 66,
 'doc_5450': 66,
 'doc_68

### ANN search

In [13]:
from ranx import Run

run_dict = defaultdict(dict)
for id, row in queries_df.iterrows():
    query_id = f"query_{id}"
    
    results = client.search(
        collection_name="wands-products",
        query_vector=models.NamedVector(
            name="product_name", 
            vector=row["query_embedding"]
        ),
        with_vectors=False,
        with_payload=False,
        limit=100,
        search_params=models.SearchParams(
            exact=False,  # disable exact search
        ),
    )
    
    for point in results:
        document_id = f"doc_{point.id}"
        run_dict[query_id][document_id] = point.score

initial_run = Run(
    run_dict, 
    name="initial",
)
initial_run['query_0']

{'doc_7465': 0.9198917,
 'doc_9234': 0.8231317,
 'doc_42329': 0.8180745,
 'doc_24010': 0.8144921,
 'doc_18273': 0.8132366,
 'doc_18276': 0.8011744,
 'doc_25431': 0.80087614,
 'doc_18272': 0.7891395,
 'doc_36910': 0.78862727,
 'doc_18277': 0.7806532,
 'doc_19456': 0.7738905,
 'doc_40996': 0.767735,
 'doc_24006': 0.76630574,
 'doc_18274': 0.75972587,
 'doc_18275': 0.7578186,
 'doc_24008': 0.7575546,
 'doc_18270': 0.75735724,
 'doc_24009': 0.75672746,
 'doc_26069': 0.75535953,
 'doc_42330': 0.7552161,
 'doc_31556': 0.75213504,
 'doc_4410': 0.7512182,
 'doc_26070': 0.7457843,
 'doc_4034': 0.7441742,
 'doc_6168': 0.7408839,
 'doc_7506': 0.74034566,
 'doc_28058': 0.7397114,
 'doc_18271': 0.7395574,
 'doc_26068': 0.73572487,
 'doc_15612': 0.7324223,
 'doc_18158': 0.7324223,
 'doc_12409': 0.7313613,
 'doc_6982': 0.7313613,
 'doc_28687': 0.7313613,
 'doc_33689': 0.7294611,
 'doc_39461': 0.72925216,
 'doc_251': 0.72697634,
 'doc_2187': 0.720437,
 'doc_33690': 0.71746707,
 'doc_31557': 0.71540356

In [32]:
from ranx import evaluate

evaluate(
    qrels=qrels, 
    run=initial_run, 
    metrics=["precision@25", "precision@50", "precision@100"]
)

{'precision@25': 0.9979166666666667,
 'precision@50': 0.9954583333333334,
 'precision@100': 0.9296666666666666}

## Tweaking the HNSW parameters

In [16]:
client.update_collection(
    collection_name="wands-products",
    hnsw_config=models.HnswConfigDiff(
        m=64, 
        ef_construct=200,
    )
)

True

In [17]:
import time

time.sleep(1.0)
collection = client.get_collection("wands-products")
while collection.status != models.CollectionStatus.GREEN:
    time.sleep(1.0)
    collection = client.get_collection("wands-products")
    
collection

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=85988, points_count=42994, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'product_description': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), 'product_name': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=64, ef_construct=200, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=2, max_segment_size=None, memmap_thresh

In [18]:
tweaked_run_dict = defaultdict(dict)
for id, row in queries_df.iterrows():
    query_id = f"query_{id}"
    
    results = client.search(
        collection_name="wands-products",
        query_vector=models.NamedVector(
            name="product_name", 
            vector=row["query_embedding"]
        ),
        with_vectors=False,
        with_payload=False,
        limit=100,
        search_params=models.SearchParams(
            exact=False,  # disable exact search
        ),
    )
    
    for point in results:
        document_id = f"doc_{point.id}"
        tweaked_run_dict[query_id][document_id] = point.score
    
tweaked_run = Run(
    tweaked_run_dict, 
    name="tweaked"
)
tweaked_run['query_0']

{'doc_7465': 0.9198917,
 'doc_9234': 0.8231317,
 'doc_42329': 0.8180745,
 'doc_24010': 0.8144921,
 'doc_18273': 0.8132366,
 'doc_18276': 0.8011744,
 'doc_25431': 0.80087614,
 'doc_18272': 0.7891395,
 'doc_36910': 0.78862727,
 'doc_18277': 0.7806532,
 'doc_19456': 0.7738905,
 'doc_40996': 0.767735,
 'doc_24006': 0.76630574,
 'doc_18274': 0.75972587,
 'doc_18275': 0.7578186,
 'doc_24008': 0.7575546,
 'doc_18270': 0.75735724,
 'doc_24009': 0.75672746,
 'doc_26069': 0.75535953,
 'doc_42330': 0.7552161,
 'doc_31556': 0.75213504,
 'doc_4410': 0.7512182,
 'doc_26070': 0.7457843,
 'doc_4034': 0.7441742,
 'doc_6168': 0.7408839,
 'doc_7506': 0.74034566,
 'doc_28058': 0.7397114,
 'doc_18271': 0.7395574,
 'doc_26068': 0.73572487,
 'doc_18158': 0.7324223,
 'doc_15612': 0.7324223,
 'doc_6982': 0.7313613,
 'doc_12409': 0.7313613,
 'doc_28687': 0.7313613,
 'doc_33689': 0.7294611,
 'doc_39461': 0.72925216,
 'doc_251': 0.72697634,
 'doc_2187': 0.720437,
 'doc_33690': 0.71746707,
 'doc_31557': 0.71540356

In [28]:
i = 0
for run1, run2 in zip(initial_run['query_0'].items(), tweaked_run['query_0'].items()):
    if run1 != run2:
        print(i, run1, run2)
    i += 1

29 ('doc_15612', 0.7324223) ('doc_18158', 0.7324223)
30 ('doc_18158', 0.7324223) ('doc_15612', 0.7324223)
31 ('doc_12409', 0.7313613) ('doc_6982', 0.7313613)
32 ('doc_6982', 0.7313613) ('doc_12409', 0.7313613)
54 ('doc_5450', 0.6690352) ('doc_16301', 0.6690352)
55 ('doc_16301', 0.6690352) ('doc_5450', 0.6690352)
71 ('doc_6170', 0.647848) ('doc_209', 0.647848)
72 ('doc_209', 0.647848) ('doc_6170', 0.647848)
89 ('doc_26833', 0.6365688) ('doc_1148', 0.63802826)
90 ('doc_25132', 0.63557523) ('doc_26833', 0.6365688)
91 ('doc_8593', 0.63368416) ('doc_25132', 0.63557523)
93 ('doc_14794', 0.63361037) ('doc_8593', 0.63368416)
94 ('doc_31397', 0.63336325) ('doc_14794', 0.63361037)
95 ('doc_1059', 0.6332264) ('doc_31397', 0.63336325)
96 ('doc_14392', 0.6327452) ('doc_1059', 0.6332264)
97 ('doc_18504', 0.63244104) ('doc_14392', 0.6327452)
98 ('doc_1372', 0.63219) ('doc_18504', 0.63244104)
99 ('doc_8130', 0.63218194) ('doc_1372', 0.63219)


In [33]:
evaluate(
    qrels=qrels, 
    run=tweaked_run, 
    metrics=["precision@25", "precision@50", "precision@100"]
)

{'precision@25': 1.0, 'precision@50': 1.0, 'precision@100': 0.9860625000000002}